<a href="https://colab.research.google.com/github/manojmulammagari/Fashion-Vision-PyTorch/blob/main/Fashion_Vision_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [4]:
# 1. Define the Dataset
train_data = datasets.FashionMNIST(
    root = 'data',
    download=True,
    train = True,
    transform = ToTensor()
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 23.8MB/s]


In [5]:
test_data = datasets.FashionMNIST(
    root = 'data',
    download=True,
    train = False,
    transform = ToTensor()
)

In [6]:
# 2. Define the DataLoader
train_loader =DataLoader(dataset=train_data, batch_size=64,shuffle=True)
test_loader =DataLoader(dataset=test_data, batch_size=64,shuffle=False)

print(f"Number of training batches : {len(train_loader)}")
print(f"Number of testing batches : ",{len(test_loader)})

Number of training batches : 938
Number of testing batches :  {157}


In [7]:
import torch.nn as nn
class FashionCNN(nn.Module):
  """
  A robust CNN architecture for 28 X 28 grayscale image classification.
  """
  def __init__(self,num_classes=10):
    super(FashionCNN,self).__init__()

    # Block 1: Initial Feature Extraction (Edges, corners)
    self.conv_block1 = nn.Sequential(
        nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    # Block 2: Deep Feature Extraction (Shapes, clothing parts)
    self.conv_block2 = nn.Sequential(
        nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )
    #Block 3 : Classification Head
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=64*7*7,out_features=128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(in_features=128, out_features=num_classes)
    )
  def forward(self,x):
    x = self.conv_block1(x)
    x = self.conv_block2(x)
    out = self.classifier(x)
    return out

# Instantiate the model to verify it works
model = FashionCNN(num_classes=10)
print(model)

FashionCNN(
  (conv_block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [8]:
import torch.optim as optim

# 1. Set up Hardware (Move to GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Training on device: {device}")

# 2. Define the Loss Function (The Grader)
criterion = nn.CrossEntropyLoss()

# 3. Define the Optimizer (The Study Strategy)
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training on device: cuda


In [13]:
epochs = 20  # Number of full passes through the dataset

for epoch in range(epochs):
    model.train()  # Put the model in training mode
    running_loss = 0.0

    # Loop through batches of data from the delivery truck
    for images, labels in train_loader:

        # 1. Send data to the same hardware as the model
        images, labels = images.to(device), labels.to(device)

        # 2. Clear old memory
        optimizer.zero_grad()

        # 3. Forward Pass (The Student takes the test)
        outputs = model(images)

        # 4. Calculate Loss (The Teacher grades the test)
        loss = criterion(outputs, labels)

        # 5. Backward Pass (Calculate how to fix the mistakes)
        loss.backward()

        # 6. Update Weights (The Student learns!)
        optimizer.step()

        running_loss += loss.item()

    # Print the average loss for this epoch
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Training Loss: {epoch_loss:.4f}")

Epoch [1/20] - Training Loss: 0.1949
Epoch [2/20] - Training Loss: 0.1781
Epoch [3/20] - Training Loss: 0.1672
Epoch [4/20] - Training Loss: 0.1552
Epoch [5/20] - Training Loss: 0.1436
Epoch [6/20] - Training Loss: 0.1330
Epoch [7/20] - Training Loss: 0.1276
Epoch [8/20] - Training Loss: 0.1175
Epoch [9/20] - Training Loss: 0.1106
Epoch [10/20] - Training Loss: 0.1053
Epoch [11/20] - Training Loss: 0.1009
Epoch [12/20] - Training Loss: 0.0947
Epoch [13/20] - Training Loss: 0.0908
Epoch [14/20] - Training Loss: 0.0857
Epoch [15/20] - Training Loss: 0.0816
Epoch [16/20] - Training Loss: 0.0771
Epoch [17/20] - Training Loss: 0.0771
Epoch [18/20] - Training Loss: 0.0741
Epoch [19/20] - Training Loss: 0.0718
Epoch [20/20] - Training Loss: 0.0664


In [14]:
# 1. Put the model in evaluation mode
model.eval()

correct_predictions = 0
total_samples = 0

# 2. Turn off the Autograd engine
with torch.no_grad():

    # 3. Loop over the testing delivery truck
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # 4. The student takes the final exam
        outputs = model(images)

        # 5. Get the highest probability prediction
        _, predicted = torch.max(outputs.data, dim=1)

        # 6. Tally up the correct answers
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

# 7. Calculate and print the final accuracy
accuracy = 100 * correct_predictions / total_samples
print(f"Final Test Accuracy on 10,000 unseen images: {accuracy:.2f}%")

Final Test Accuracy on 10,000 unseen images: 92.43%
